# Indonesian Spam Email Detection: Preprocessing

Minimal text preprocessing for the Indonesian email spam dataset (`email_spam_indo.csv`, columns `Kategori` and `Pesan`).

**Steps**
1. Handle missing or empty email text
2. Convert to lowercase
3. Normalize repeated characters, whitespace, and common Indonesian spellings
4. Tokenize into words
5. Remove Indonesian stopwords (NLTK list)

**Requirements:** `pip install -r requirements.txt` (pandas, nltk)  
**Input:** `email_spam_indo.csv` in the same folder as this notebook  
**Output:** `email_spam_indo_clean.csv`

In [1]:
import re
from pathlib import Path

import pandas as pd
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("indonesian"))
print("Indonesian stopwords loaded:", len(STOPWORDS))

Indonesian stopwords loaded: 757


## 1. Load data

In [2]:
DATA_PATH = Path("email_spam_indo.csv")
OUTPUT_PATH = Path("email_spam_indo_clean.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df["Kategori"].value_counts())
df.head()

Shape: (2636, 2)
Kategori
spam    1368
ham     1268
Name: count, dtype: int64


,Kategori,Pesan
0,spam,Secara alami tak tertahankan identitas perusah...
1,spam,Fanny Gunslinger Perdagangan Saham adalah Merr...
2,spam,Rumah -rumah baru yang luar biasa menjadi muda...
3,spam,4 Permintaan Khusus Pencetakan Warna Informasi...
4,spam,"Jangan punya uang, dapatkan CD perangkat lunak..."


## 2. Preprocessing function

The lowercase step can be removed if case turns out to matter for tone (see the comment in the function).

In [3]:
# Small map of common Indonesian slang / short spellings
NORMALIZE = {
    "yg": "yang", "dgn": "dengan", "utk": "untuk", "tdk": "tidak", "gak": "tidak",
    "ga": "tidak", "nggak": "tidak", "krn": "karena", "dr": "dari", "sdh": "sudah",
    "blm": "belum", "bgt": "banget", "jgn": "jangan", "tp": "tapi", "dlm": "dalam",
    "klo": "kalau", "kalo": "kalau", "sy": "saya", "org": "orang", "dg": "dengan",
}

def preprocess(text):
    # Handle missing / empty text
    if not isinstance(text, str) or not text.strip():
        return ""
    # Lowercase (remove this line if case matters for tone)
    text = text.lower()
    # Normalize repeated characters ("haiiii" -> "haii") and whitespace
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[^a-z\s]", " ", text)  # keep letters only
    text = re.sub(r"\s+", " ", text).strip()
    # Tokenize
    tokens = text.split()
    # Normalize spelling, then remove stopwords and single-letter tokens
    tokens = [NORMALIZE.get(t, t) for t in tokens]
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return " ".join(tokens)

# Quick sanity check
preprocess("Haiiiii   gratisss  yg   bgt!!")

'haii gratiss banget'

## 3. Apply to the dataset

In [4]:
df["Pesan"] = df["Pesan"].fillna("")
df["text_clean"] = df["Pesan"].apply(preprocess)

before = len(df)
df = df[df["text_clean"] != ""]                                # drop rows empty after cleaning
df = df.drop_duplicates(subset=["Kategori", "text_clean"])     # drop duplicates
print(f"Rows: {before} -> {len(df)}")
print(df["Kategori"].value_counts())

Rows: 2636 -> 2617
Kategori
spam    1367
ham     1250
Name: count, dtype: int64


## 4. Before vs. after

In [5]:
for _, row in df.sample(3, random_state=42).iterrows():
    print(f"[{row['Kategori']}]")
    print("BEFORE:", row["Pesan"][:200])
    print("AFTER: ", row["text_clean"][:200])
    print()

[ham]
BEFORE: Pendahuluan untuk buku Vince ,? Semoga Anda baik -baik saja. ? Kami berbicara beberapa waktu lalu tentang siapa yang harus menulis kata pengantar untuk buku ini, dan Anda dengan ramah menawarkan bahwa
AFTER:  pendahuluan buku vince semoga berbicara menulis pengantar buku ramah menawarkan menyediakan menyadari sibuk chris les melanjutkan menulis meninjau mengubah menulis pendahuluan dihargai pendapat terima

[spam]
BEFORE: Basis data yang tidak ingin Anda ketahui oleh Bill Gates! ! ! ! ! PEMBERITAHUAN PENTING: Mengenai nama domain Anda * jika Anda memiliki a. com /. net /. org, Anda disarankan untuk mendaftarkan Anda. D
AFTER:  basis data ketahui bill gates pemberitahuan nama domain memiliki com net orang disarankan mendaftarkan domain situs web ws orang mengambilnya perusahaan yahoo att intel mendaftarkan domain situs web w

[ham]
BEFORE: Re: Re: Workshop Ini adalah berita bagus bagi saya dan terlebih lagi untuk ICBI (atau sebaliknya.). Bisakah Anda menelepon saya hari in

## 5. Save

In [6]:
df[["Kategori", "text_clean"]].to_csv(OUTPUT_PATH, index=False)
print("Saved to", OUTPUT_PATH)

Saved to email_spam_indo_clean.csv
